# 07 Spatial Integration - Zonal Statistics

**Date:** 2026-04-19 (Updated)  
**Objective:** Integrate soil, NDVI, and weather data at the field level using zonal statistics

## Summary

This notebook performs zonal statistics to integrate multi-source geospatial data:

1. **Soil Data** - Field-level aggregated soil properties (pH, OM, AWC, drainage) for 24 fields
2. **NDVI Raster** - Extract vegetation indices per field polygon (top 10 fields)
3. **Weather Data** - Seasonal aggregations per field (temperature, precipitation, GDD) for 10 fields

Output: Unified field-level summary with all integrated data sources.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from rasterstats import zonal_stats
import rasterio

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

DATA_DIR = Path("/workspaces/my-farm-advisor/data/workspace")
OUTPUT_DIR = DATA_DIR / "output" / "assignment-07"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Field Boundaries

In [ ]:
fields = gpd.read_file(DATA_DIR / "NC_field_boundaries_EPSG4326_2026-04-01.geojson")
fields["centroid"] = fields.geometry.centroid
fields["lat"] = fields.centroid.y
fields["lon"] = fields.centroid.x

print(f"Loaded {len(fields)} fields")
display(fields[["field_id", "county_name", "area_acres", "lat", "lon"]])

## 2. Load and Aggregate Soil Data

In [ ]:
soil = pd.read_csv(DATA_DIR / "NC_soil_crop_data_EPSG4326_2026-04-01.csv")
print(f"Loaded {len(soil)} soil records")
print(f"Columns: {list(soil.columns)}")

In [ ]:
soil_agg = soil.groupby("field_id").agg(
    mukey_count=("mukey", "nunique"),
    drainage_primary=("drainagecl", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
    om_mean=("om_r", "mean"),
    om_min=("om_r", "min"),
    om_max=("om_r", "max"),
    ph_mean=("ph1to1h2o_r", "mean"),
    ph_min=("ph1to1h2o_r", "min"),
    ph_max=("ph1to1h2o_r", "max"),
    awc_mean=("awc_r", "mean"),
    clay_mean=("claytotal_r", "mean"),
    sand_mean=("sandtotal_r", "mean"),
    cec_mean=("cec7_r", "mean"),
).reset_index()

soil_agg = soil_agg.round(2)
print(f"Aggregated to {len(soil_agg)} fields")
display(soil_agg)

## 3. Zonal Statistics from NDVI Raster
Note: NDVI rasters available for top 10 fields from ndvi_summary_top10.csv

In [ ]:
ndvi_path = DATA_DIR / "output" / "assignment-05" / "osm-260949778_NDVI_EPSG4326.tif"
print(f"NDVI raster: {ndvi_path}")
print(f"Exists: {ndvi_path.exists()}")

with rasterio.open(ndvi_path) as src:
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Shape: {src.shape}")

In [ ]:
fields_in_bounds = fields[
    (fields.geometry.centroid.x >= -82.0) & (fields.geometry.centroid.x <= -79.0) &
    (fields.geometry.centroid.y >= 34.5) & (fields.geometry.centroid.y <= 36.5)
].copy()

print(f"Fields potentially overlapping NDVI extent: {len(fields_in_bounds)}")

stats = zonal_stats(
    fields_in_bounds,
    ndvi_path,
    stats=["mean", "std", "min", "max", "median", "count"],
    prefix="ndvi_"
)

ndvi_df = pd.DataFrame(stats)
ndvi_df["field_id"] = fields_in_bounds["field_id"].values

print(f"Extracted NDVI stats for {len(ndvi_df)} fields (top 10 fields with NDVI data)")
display(ndvi_df)

## 4. Load and Aggregate Weather Data

In [ ]:
weather = pd.read_csv(DATA_DIR / "output" / "assignment-06" / "weather_top10_2020_2024.csv")
weather["date"] = pd.to_datetime(weather["date"])
weather["year"] = weather["date"].dt.year
weather["month"] = weather["date"].dt.month
weather["season"] = weather["month"].apply(lambda m: "spring" if m in [3,4,5] else "summer" if m in [6,7,8] else "fall" if m in [9,10,11] else "winter")

print(f"Loaded {len(weather)} daily weather records")
print(f"Fields: {weather['field_id'].unique()}")
print(f"Years: {sorted(weather['year'].unique())}")

In [ ]:
def calc_gdd(row, base=10, cap=30):
    t_avg = ((row["T2M_MIN"] + row["T2M_MAX"]) / 2)
    t_adj = min(max(t_avg, base), cap)
    return max(t_adj - base, 0)

weather["gdd"] = weather.apply(calc_gdd, axis=1)

weather_growing = weather[(weather["month"] >= 4) & (weather["month"] <= 9)]

weather_agg = weather_growing.groupby("field_id").agg(
    temp_mean=("T2M", "mean"),
    temp_max_mean=("T2M_MAX", "mean"),
    temp_min_mean=("T2M_MIN", "mean"),
    precip_total=("PRECTOTCORR", "sum"),
    precip_days=("PRECTOTCORR", lambda x: (x > 0).sum()),
    gdd_total=("gdd", "sum"),
    solar_total=("ALLSKY_SFC_SW_DWN", "sum"),
    years=("year", "nunique")
).reset_index()

weather_agg = weather_agg.round(2)
print(f"Aggregated weather for {len(weather_agg)} fields (growing season Apr-Sep)")
display(weather_agg)

## 5. Integrate All Data Sources

In [ ]:
integrated = fields[["field_id", "county_name", "area_acres", "lat", "lon"]].copy()

integrated = integrated.merge(soil_agg, on="field_id", how="left")
integrated = integrated.merge(ndvi_df, on="field_id", how="left")
integrated = integrated.merge(weather_agg, on="field_id", how="left")

print(f"Integrated {len(integrated)} fields")
display(integrated)

In [ ]:
output_csv = OUTPUT_DIR / "zonal_integration_summary.csv"
integrated.to_csv(output_csv, index=False)
print(f"Saved to {output_csv}")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

ax1 = axes[0, 0]
ax1.barh(integrated["field_id"], integrated["om_mean"], color="brown")
ax1.set_xlabel("Organic Matter (%)")
ax1.set_title("Mean Organic Matter by Field")

ax2 = axes[0, 1]
ax2.barh(integrated["field_id"], integrated["ph_mean"], color="green")
ax2.set_xlabel("pH")
ax2.set_title("Mean pH by Field")
ax2.axvline(6.5, color="grey", linestyle="--", alpha=0.5, label="Optimal")
ax2.axvline(5.5, color="grey", linestyle="--", alpha=0.5)

ax3 = axes[1, 0]
if "ndvi_mean" in integrated.columns:
    ndvi_data = integrated[integrated["ndvi_mean"].notna()]
    if len(ndvi_data) > 0:
        ax3.barh(ndvi_data["field_id"], ndvi_data["ndvi_mean"], color="darkgreen")
        ax3.set_xlabel("NDVI Mean")
        ax3.set_title("Mean NDVI by Field")
    else:
        ax3.text(0.5, 0.5, "No NDVI data available", ha="center")
else:
    ax3.text(0.5, 0.5, "No NDVI data available", ha="center")

ax4 = axes[1, 1]
ax4.barh(integrated["field_id"], integrated["gdd_total"] / integrated["years"], color="orange")
ax4.set_xlabel("Annual GDD (growing season)")
ax4.set_title("Mean GDD by Field")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "zonal_stats_summary.png", dpi=150)
print(f"Saved {OUTPUT_DIR / 'zonal_stats_summary.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(integrated["om_mean"], integrated["gdd_total"], s=100, alpha=0.7)
for i, row in integrated.iterrows():
    ax.annotate(row["field_id"][:10], (row["om_mean"], row["gdd_total"]), fontsize=8)
ax.set_xlabel("Organic Matter (%)")
ax.set_ylabel("Growing Season GDD")
ax.set_title("Organic Matter vs. GDD by Field")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "om_vs_gdd.png", dpi=150)
print(f"Saved {OUTPUT_DIR / 'om_vs_gdd.png'}")

### Spatial Join Visualization

In [ ]:
spatial_join_path = OUTPUT_DIR / "spatial_join_5largest.png"
if spatial_join_path.exists():
    img = plt.imread(spatial_join_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Spatial Join - Top 5 Largest Fields")
    plt.tight_layout()
    plt.show()
    print(f"Loaded spatial join visualization")
else:
    print(f"Spatial join image not found at {spatial_join_path}")

## 7. Data Coverage Summary

In [ ]:
coverage = pd.DataFrame({
    "field_id": integrated["field_id"],
    "soil": integrated[["om_mean", "ph_mean", "awc_mean"]].notna().all(axis=1),
    "ndvi": integrated["ndvi_mean"].notna() if "ndvi_mean" in integrated.columns else False,
    "weather": integrated["gdd_total"].notna()
})

display(coverage)
print(f"\nFields with all three sources: {coverage[['soil', 'ndvi', 'weather']].all(axis=1).sum()}")